# Asthma incidents from GP and HES APC only

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
pd.set_option('display.max_rows', 500)

In [ ]:


root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)

In [ ]:
from env.parameters import P


In [ ]:

from phenotyping.codebase_phenotyping import (
pheno_all_evdt_extractor,
pheno_rank_and_filter,
pheno_keep_one_row,
clean_biobank_ado_pheno,
join_gp_hes_biobank_single_row_dfs,
join_single_row_dfs
)
from util.parquet_maker import dask_to_parquet
from util.general_utils import field_availability_check, list_field_instances
from util.dask_utils import import_field_from_dask
from util.parquet_maker import dask_to_parquet

In [ ]:
import dask.dataframe as dd



# Load main files

In [ ]:
# cohort
df_cohort = pd.read_csv(f'''{P.output_csv_path}/cohort_2_advanced.csv''')
df_cohort.head()

In [ ]:
# GP data
dd_gp = dd.read_parquet(f'''{P.output_parquet_path}/gp_clinical_curated''')
dd_gp.head()

In [ ]:
dd_gp.dtypes

In [ ]:
df_gp = dd_gp[["eid", "read_curated", "evdt_gp"]].compute()
df_gp.dtypes

In [ ]:
df_gp.head()

In [ ]:
# HES data
df_hes = pd.read_csv(f'''{P.output_csv_path}/hesin_diag_curated.csv''')
df_hes

# Asthma

in GP, HES

In [ ]:
from phenotyping.codebase_phenotyping import __old_pheno_all_evdt_extractor


In [ ]:
# codelist
codelist_in = pd.read_csv(f'''{P.codelist_path}/curated/asthma.csv''')
codelist_in

In [ ]:
# Any medicaiton codes?
codelist_in[codelist_in["code_clean"].str.match(r'^[a-z]')]

In [ ]:
assign_pheno_name = "asthma"
print(assign_pheno_name)

## Asthma in GP

In [ ]:
out_gp_long = pheno_all_evdt_extractor(df_data_clean=df_gp,
                                             col_evdt_data="evdt_gp",
                                             col_code_data="read_curated",
                                             df_codelist=codelist_in,
                                             col_code_codelist="code_clean",
                                             col_vocab_codelist="vocab",
                                             use_vocab="Read2",
                                             col_assign_pheno_name="pheno",
                                             assign_pheno_name=assign_pheno_name,
                                             col_codelist_multicategory=None,
                                             join_type="read2")

In [ ]:
out_gp_long.head()

In [ ]:
out_gp_long = out_gp_long[['eid', 'evdt_gp', 'pheno']]
#out_gp_long["isin_"] = "gp"
out_gp_long.head(5)

In [ ]:
out_gp_long.shape[0]

In [ ]:
out_gp_long_ranked = pheno_rank_and_filter(out_gp_long, col_evdt="evdt_gp", col_eid="eid", earliest_ranks_1=True)
out_gp_long_ranked.head(10)


In [ ]:
# Use this later for joining with HES (and ADO, if any)
out_gp_single_row = pheno_keep_one_row(df_ranked_long=out_gp_long_ranked,
                                       col_row_number="row_number")
out_gp_single_row.head(10)

In [ ]:
 out_gp_single_row['eid'].count() == out_gp_single_row['eid'].nunique()

In [ ]:
P.output_phenotypes_parquet_path

In [ ]:
# Save as parquet
dd_long_gp = dd.from_pandas(out_gp_long, npartitions=5)
dask_to_parquet(dd_long_gp, output_path=P.output_phenotypes_parquet_path, output_parquet_name="asthma_gp_long", write_index=False)


## Asthma in HES

In [ ]:
df_hes_long =pheno_all_evdt_extractor(df_data_clean=df_hes,
                                             col_evdt_data="epistart",
                                             col_code_data="diag_icd10",
                                             df_codelist=codelist_in,
                                             col_code_codelist="code_clean",
                                             col_vocab_codelist="vocab",
                                             use_vocab="ICD10",
                                             col_assign_pheno_name="pheno",
                                             assign_pheno_name=assign_pheno_name,
                                             col_codelist_multicategory=None,
                                             join_type="icd10")
df_hes_long.head()

In [ ]:
df_hes_long = df_hes_long[['eid', 'epistart', 'pheno']]
#df_hes_long["isin_hes"] = 1

df_hes_long.head(5)

In [ ]:
out_hes_long_ranked = pheno_rank_and_filter(df_hes_long, col_evdt="epistart", col_eid="eid", earliest_ranks_1=True)
out_hes_long_ranked.head(10)

In [ ]:
# Use this later for joining with GP (and ADO, if any)
out_hes_single_row = pheno_keep_one_row(df_ranked_long=out_hes_long_ranked,
                                       col_row_number="row_number")
out_hes_single_row.head(10)

In [ ]:
# Save as parquet
dd_long_hes = dd.from_pandas(df_hes_long, npartitions=5)
dask_to_parquet(dd_long_hes, output_path=P.output_phenotypes_parquet_path, output_parquet_name="asthma_hes_long", write_index=False)


# Make a single GP + HES pheno and save

In [ ]:
df_gp_hes = join_gp_hes_biobank_single_row_dfs(pheno_gp=out_gp_single_row, pheno_hes=out_hes_single_row,
                                   col_evdt_gp="evdt_gp", col_evdt_hes="epistart", pheno_name=assign_pheno_name, gp_and_hes_only=True, keep_minimum=True,col_eid="eid")

In [ ]:
df_gp_hes = df_gp_hes[['eid', f'''evdt_{assign_pheno_name}''']]
#df_out[f'''pheno_{assign_pheno_name}'''] = 1
df_gp_hes.head()

In [ ]:

# Save
df_gp_hes.to_csv(f'''{P.output_phenotypes_csv_path}/incident_{assign_pheno_name}_gp_hes.csv''', index=False)

In [ ]:
P.output_phenotypes_csv_path

In [ ]:
df_gp_hes = pd.read_csv(f'''{P.output_phenotypes_csv_path}/incident_{assign_pheno_name}_gp_hes.csv''')
df_gp_hes.head()

In [ ]:
df_gp_hes[f'''evdt_{assign_pheno_name}'''] = pd.to_datetime(df_gp_hes[f'''evdt_{assign_pheno_name}'''])


In [ ]:
out_gp_single_row.head()

In [ ]:
out_hes_single_row.head()

In [ ]:
df_gp_hes.head()

In [ ]:
df_gp_hes.shape

In [ ]:

# Save
df_gp_hes.to_csv(f'''{P.output_phenotypes_csv_path}/incident_asthma_gp_hes_only.csv''', index=False)